# Ciello — estúdio do clone (Wan2GP no Kaggle, GPU grátis)

Roda o **[Wan2GP](https://github.com/deepbeepmeep/Wan2GP)** numa GPU grátis do Kaggle e abre uma tela no navegador com tudo de que o clone precisa:

| Para quê | Modelo no Wan2GP |
|---|---|
| Foto realista do personagem / de você em cenários | **Qwen Image Edit Plus 2511** (com fotos de referência) ou **Z-Image Turbo** (só texto, mais leve) |
| Personagem **falando** com a narração | **InfiniteTalk** (Single Speaker 480p) ou **LongCat Avatar 1.5** |
| Personagem **andando** / se mexendo no cenário | **Wan 2.2 TextImage2Video 5B FastWan** (480p) ou **LTX-2.5 Distilled** |
| Trocar o rosto mantendo o movimento | **Lynx** |
| Narração de teste / voz clonada | **Qwen3 TTS Base**, **OmniVoice** ou **IndexTTS 2.5** |

**Antes de rodar (uma vez):** conta no Kaggle com telefone verificado → neste notebook, *Settings* → **Accelerator: GPU T4 x2** e **Internet: On**.

**Rodar:** *Run All*. A instalação leva ~10 min. No fim aparece um link `https://….gradio.live`: abra no navegador. Cada modelo baixa sozinho na primeira vez que você usar (vários GB, demora).

**Pegar os vídeos:** tudo que você gerar é copiado a cada minuto para `/kaggle/working/saida`. Baixe pela aba *Output* do notebook e coloque em `reels/public/clone/` no repositório.

Cota grátis: ~30 h de GPU por semana, sessões de até 9 h. **Desligue a sessão (Stop) quando terminar.**

In [ ]:
# 1. GPU e espaço em disco
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | sed 1d

In [ ]:
# 2. Baixa o Wan2GP em /tmp (a pasta de trabalho do Kaggle tem só 20 GB e os modelos passam disso)
import os, subprocess
WAN2GP = "/tmp/Wan2GP"
VERSAO = ""  # deixe vazio para a versão mais nova, ou cole um commit para fixar uma versão que funcionou
if not os.path.exists(WAN2GP):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git {WAN2GP}
if VERSAO:
    !git -C {WAN2GP} checkout {VERSAO}
print("Wan2GP no commit:", subprocess.run(["git", "-C", WAN2GP, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Dependências (~5 a 10 min)
%cd /tmp/Wan2GP
!pip install -q -r requirements.txt

In [ ]:
# 4. Copia tudo que for gerado para /kaggle/working/saida (aba Output), a cada 60 s
import shutil, threading, time, pathlib
SAIDA = pathlib.Path("/kaggle/working/saida"); SAIDA.mkdir(parents=True, exist_ok=True)
def copiar_saidas():
    while True:
        for f in pathlib.Path("/tmp/Wan2GP/outputs").rglob("*"):
            if f.suffix.lower() in {".mp4", ".png", ".jpg", ".wav", ".mp3"} and not (SAIDA / f.name).exists():
                try:
                    shutil.copy2(f, SAIDA / f.name)
                except OSError:
                    pass
        time.sleep(60)
threading.Thread(target=copiar_saidas, daemon=True).start()
print("cópia automática ligada ->", SAIDA)

In [ ]:
# 5. Abre a tela. Procure abaixo a linha "Running on public URL: https://….gradio.live"
# --profile 4 = economiza VRAM (T4 de 16 GB) · --attention sdpa = funciona na T4
%cd /tmp/Wan2GP
!python wgp.py --share --profile 4 --attention sdpa

## Teste desta noite (sem rosto e sem a sua voz)

1. **Personagem:** em *Z-Image Turbo*, gere uma foto vertical: `foto realista, homem brasileiro de 35 anos, camisa social azul-marinho, escritório moderno com luz natural, olhando para a câmera, plano médio, 9:16`.
2. **Narração de teste:** em *Qwen3 TTS Base* (ou *OmniVoice*), gere ~10 s em português: `Sua equipe perde venda no WhatsApp e você nem fica sabendo.`
3. **Fala:** em *InfiniteTalk*, use a foto como imagem inicial e o áudio da etapa 2. Resolução 480p, janela de 81 quadros (padrão do modelo).
4. **Movimento:** em *Wan 2.2 TextImage2Video 5B FastWan*, use a mesma foto com o prompt `o homem caminha pela calçada de uma rua comercial, câmera acompanhando, movimento natural`, 480p, ~5 s.
5. Baixe os vídeos da aba *Output*, renomeie para `cena1-fala.mp4` e `cena2-andando.mp4`, coloque em `reels/public/clone/` e renderize:
   `cd reels && node render.mjs CloneReel ../midia/teste-clone.mp4`

Anote o tempo de cada geração. É isso que define quantos reels com clone cabem nas 30 h semanais.

**Com você (depois):** troque a etapa 1 por *Qwen Image Edit Plus 2511* com 2 a 4 fotos suas como referência, e a etapa 2 pela voz clonada (`central/voz.py` ou *Qwen3 TTS Base* com 10 s da voz dele).